# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Minimal reproduction (pretrained 30%-label USCount-Net checkpoint on the released flower-cluster test split) is blocked: both the weights (Drive file 1pHtOO1Q6SqisQv0bTXE9oPQ-aoY3MkFc) and the dataset (Drive folder 1KP8H0qIuct56hWre5GV6ZJnzwqOY3Pip) are not anonymously accessible (HTTP 401 / sign-in redirects, verified 2026-10-05 from fresh Colab runtimes via gdown, requests and embeddedfolderview). Code-only, never-executed UNVERIFIED reference notebook delivered at deliverables/p-a9eeb18ead1f19a9c2ea5e76c0b0ffb6.ipynb with a RuntimeError guard; no scientific execution performed; publisher not called.

Attempt status: blocked. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Low-annotation apple flower counting: A color-SAM enhanced and uncertainty-guided semi-supervised framework.

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-a9eeb18ead1f19a9c2ea5e76c0b0ffb6)


# USCount-Net apple flower counting — pretrained-checkpoint evaluation (CODE-ONLY / NOT EXECUTED)

**Paper:** *Low-annotation apple flower counting: A color-SAM enhanced and uncertainty-guided semi-supervised framework.* Plant Phenomics (2026), DOI [10.1016/j.plaphe.2026.100190](https://doi.org/10.1016/j.plaphe.2026.100190) — PMC13316440 (CC BY).

**Author code:** [haohuihui5019/USCount-Net](https://github.com/haohuihui5019/USCount-Net) @ commit `19f34e9539609346cfbd515c1d7c704800dd179a`.

## ⚠️ EXECUTION BLOCKED — required materials are not publicly accessible (EN)

This notebook is an **UNVERIFIED / INCOMPLETE reference** and was **never executed**. The two materials required for the planned minimal reproduction are hosted on Google Drive and were **not reachable anonymously** (verified 2026-10-05 from a fresh Colab runtime via multiple routes):

- Pretrained weights (30% labeling ratio), Drive file `1pHtOO1Q6SqisQv0bTXE9oPQ-aoY3MkFc`: `https://drive.google.com/file/d/.../view` returns **HTTP 401**; `gdown` cannot retrieve the public link.
- Apple flower cluster dataset (Test data), Drive folder `1KP8H0qIuct56hWre5GV6ZJnzwqOY3Pip`: redirects to a **Google sign-in page**; the anonymous `embeddedfolderview` returns **401**.

The links appear not to be shared as “Anyone with the link”. **If the owner makes them public (or an authorized mirror exists), this notebook can be run top-to-bottom as written** — it contains the full faithful inference implementation from the authors' `test.py`. No results in this notebook are generated evidence.

## ⚠️ 実行ブロック — 必要な素材が公開されていない (JA)

本ノートブックは**未検証・未実行の参考実装**です。事前学習済み重み(30%ラベル比)と apple flower cluster データセット(Test data)はいずれも Google Drive 上にありますが、匿名アクセスでは取得できませんでした(401 / サインイン画面へのリダイレクト、2026-10-05 に新規 Colab ランタイムから複数経路で確認)。オーナーがリンクを公開設定にすれば、本書き込みのまま先頭から実行できます。このノートブック内に生成された結果は一切ありません。

## Scope

Planned (not executed) demonstration: evaluate the released pretrained USCount-Net checkpoint (30% labeling ratio) on the test split of the released flower-cluster dataset with the authors' `test.py` procedure, reproducing per-stage MAE/RMSE (paper Table 1 at 30% labels: bud 7.26 / anthesis 8.90 / petal-fall 6.41). Training, the semi-supervised framework, and CSAM-FE preprocessing are **not released** by the authors and are out of scope.

### Runtime selection

**Recommended: T4 GPU** (matches the authors' `test.py`, which uses CUDA). **CPU also works** for this inference-only task: inputs are 224×224 flower-cluster crops and the test split is ≈900 images, so a CPU run is slower (roughly 5–15 min estimated) but scientifically identical. The code below auto-detects the device. Select *Runtime ▸ Change runtime type* before *Run all*. Expected total time after materials are available: ~5 min setup/download + evaluation. 推奨ランタイムは T4 GPU、CPU でも推論のみなら実行可能です。

### Execution guard

This first cell intentionally stops *Run all* while the required Drive materials are not publicly accessible, so that no misleading partial result can be produced. 必要素材が公開されていない間、誤った結果を防ぐため最初のセルで実行を停止します。

In [ ]:
raise RuntimeError(
    "CODE-ONLY / NOT EXECUTED: the pretrained USCount-Net weights (Drive file "
    "1pHtOO1Q6SqisQv0bTXE9oPQ-aoY3MkFc) and the flower-cluster test data "
    "(Drive folder 1KP8H0qIuct56hWre5GV6ZJnzwqOY3Pip) were not anonymously "
    "accessible when this notebook was authored (HTTP 401 / sign-in redirect, "
    "checked 2026-10-05). Ask the repository owner to share them as 'Anyone "
    "with the link', then remove this guard and run the notebook."
)

### 1. Dependencies

`gdown` fetches the Drive weights/dataset and `timm` provides `trunc_normal_`, which the authors' `model/model.py` imports at module load. torch/torchvision are preinstalled on Colab. Colab の事前インストール済み torch を利用し、追加で gdown と timm を入れます。

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "timm"], check=True)

### 2. Download the pretrained checkpoint (30% labeling ratio)

Author-provided weights from the repo README, fetched with `gdown` into `/content`. The file must be a valid torch checkpoint; the assert fails clearly on an HTML error page. README記載の Google Drive から重みを取得し、不正な HTML でないことを確認します。

In [ ]:
import gdown, os
WEIGHT_ID = "1pHtOO1Q6SqisQv0bTXE9oPQ-aoY3MkFc"
WEIGHT_PATH = "/content/uscount_net_30pct.pth"
gdown.download(id=WEIGHT_ID, output=WEIGHT_PATH, quiet=False)
size = os.path.getsize(WEIGHT_PATH)
print("checkpoint bytes:", size)
assert size > 1_000_000, "Checkpoint download looks invalid (too small; likely an HTML error page)."

### 3. Download the flower-cluster dataset (Test data)

The Drive folder contains `Test data` and `Train data`; only the **Test** split is needed for evaluation. `gdown.download_folder` retrieves it into `/content/dataset`. The folder layout is verified at runtime; the authors' `Crowd` dataset class is used unchanged, so the evaluation follows the authors' exact data contract. Drive フォルダから Test データのみ取得します(レイアウトは実行時に確認)。

In [ ]:
import gdown
DATASET_FOLDER_ID = "1KP8H0qIuct56hWre5GV6ZJnzwqOY3Pip"
gdown.download_folder(id=DATASET_FOLDER_ID, output="/content/dataset", quiet=False, use_cookies=False)

### 4. Get the pinned author code and its dataset/model classes

Partial clone of the authors' repository at the verified commit `19f34e9…` (blobless, shallow), checked out in detached HEAD with an assert. We import the authors' own `Crowd` dataset and `vgg19` model classes unchanged rather than re-implementing them. 著者リポジトリを指定コミットで取得し、著者の `Crowd`/`vgg19` をそのまま利用します。

In [ ]:
import subprocess, sys, os
REPO_URL = "https://github.com/haohuihui5019/USCount-Net"
PINNED_COMMIT = "19f34e9539609346cfbd515c1d7c704800dd179a"
if not os.path.isdir("/content/USCount-Net"):
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    REPO_URL, "/content/USCount-Net"], check=True)
subprocess.run(["git", "-C", "/content/USCount-Net", "fetch", "--depth", "1",
                "origin", PINNED_COMMIT], check=True)
subprocess.run(["git", "-C", "/content/USCount-Net", "checkout", "FETCH_HEAD"], check=True)
head = subprocess.run(["git", "-C", "/content/USCount-Net", "rev-parse", "HEAD"],
                      check=True, capture_output=True, text=True).stdout.strip()
assert head == PINNED_COMMIT, f"unexpected HEAD {head}"
sys.path.insert(0, "/content/USCount-Net")
print("author repo pinned at", head)

### 5. Build the model and load the checkpoint (device-adaptive)

`vgg19(25)` follows the authors' `test.py` exactly; the constructor loads the ImageNet VGG-19 backbone via `torch.utils.model_zoo`. The only adaptation to the authors' script is device selection (`cuda` if available, else `cpu`) — the authors' `test.py` hardcodes `cuda`. 著者の `test.py` に従い `vgg19(25)` を構築。唯一の変更はデバイス自動選択です。

In [ ]:
import torch
from model.model import vgg19

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
model = vgg19(25)
checkpoint = torch.load(WEIGHT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["state_dict"])
model.to(device).eval()
print("state_dict loaded:", len(checkpoint["state_dict"]), "tensors")

### 6. Evaluate MAE / RMSE on the test split (faithful port of `test.py`)

Batch size 1, `Crowd(data_dir, 224, 8, method='val')`, three-channel density output; per-stage predictions are channel sums compared with `gt_counts` exactly as in the authors' loop, then MAE and RMSE (printed there as `MSE`) per stage and overall. `DATA_DIR` is located from the downloaded Drive folder — adjust the glob if the layout differs. 著者の評価ループを忠実に再現し、stage 別 MAE/RMSE を計算します。

In [ ]:
import numpy as np
from torch.utils.data import DataLoader
from datasets.dataset import Crowd
import glob

candidates = sorted(glob.glob("/content/dataset/**/Test*", recursive=True))
print("Test-data candidates:", candidates)
DATA_DIR = candidates[0] if candidates else "/content/dataset"

dataset = Crowd(DATA_DIR, 224, 8, method="val")
dataloader = DataLoader(dataset, 1, shuffle=False, pin_memory=False)
print("test images:", len(dataset))

epoch_res, epoch_res_bud, epoch_res_bloom, epoch_res_faded = [], [], [], []
with torch.no_grad():
    for inputs, gt_counts, name in dataloader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        pred_bud, pred_bloom, pred_faded = outputs[:, 0:1], outputs[:, 1:2], outputs[:, 2:3]
        epoch_res.append(sum(gt_counts) - torch.sum(outputs).item())
        epoch_res_bud.append(gt_counts[0] - torch.sum(pred_bud).item())
        epoch_res_bloom.append(gt_counts[1] - torch.sum(pred_bloom).item())
        epoch_res_faded.append(gt_counts[2] - torch.sum(pred_faded).item())

def mae(v):
    return float(np.mean(np.abs(np.array(v))))

def rmse(v):
    return float(np.sqrt(np.mean(np.square(np.array(v)))))

print(f"MAE overall {mae(epoch_res):.2f} | bud {mae(epoch_res_bud):.2f} | "
      f"anthesis {mae(epoch_res_bloom):.2f} | petal-fall {mae(epoch_res_faded):.2f}")
print(f"RMSE overall {rmse(epoch_res):.2f} | bud {rmse(epoch_res_bud):.2f} | "
      f"anthesis {rmse(epoch_res_bloom):.2f} | petal-fall {rmse(epoch_res_faded):.2f}")

### 7. Comparison with the paper's reported results

The computed values are compared with Table 1 of the paper (30% labeling ratio: bud MAE 7.26, anthesis 8.90, petal-fall 6.41). These paper numbers are **quoted reference values, not outputs of this notebook** (which was never executed). Small deviations are expected from the released checkpoint and split randomness. 論文 Table 1 の数値は引用した参耀値であり、本ノートブックの出力ではありません。

In [ ]:
import pandas as pd
paper_30pct = {"bud": 7.26, "anthesis": 8.90, "petal-fall": 6.41}  # paper Table 1, quoted
rows = [{"stage": k,
         "paper MAE (30% labels, quoted)": v,
         "this run MAE": m} for (k, v), m in zip(paper_30pct.items(),
        [mae(epoch_res_bud), mae(epoch_res_bloom), mae(epoch_res_faded)])]
pd.DataFrame(rows)

### 8. Input preview and density-map visualization (planned, not executed)

Show one test cluster image and its predicted three-stage density maps. 評価に使う入力画像と予測密度マップの可視化(未実行)。

In [ ]:
import matplotlib.pyplot as plt
inputs, gt_counts, name = next(iter(dataloader))
with torch.no_grad():
    out = model(inputs.to(device))[0].cpu().numpy()
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
img = inputs[0].permute(1, 2, 0).numpy()
img = (img - img.min()) / (img.max() - img.min() + 1e-8)
axes[0].imshow(img); axes[0].set_title(f"input cluster {name[0]}")
for ax, ch, t in zip(axes[1:], range(3), ["bud", "anthesis", "petal-fall"]):
    ax.imshow(out[ch], cmap="jet"); ax.set_title(f"predicted density: {t} (sum={out[ch].sum():.1f})")
plt.tight_layout(); plt.savefig("/content/uscountnet_prediction.png", dpi=120); plt.show()

## Limitations and next steps

- Training, the semi-supervised (Mean-Teacher/FFUE/PLF) framework and CSAM-FE preprocessing are **not released** by the authors (“to be released”); this notebook covers inference only.
- No license file is present in the repository; reuse terms must be confirmed with the authors.
- To activate this notebook: make both Drive resources public (or supply an authorized mirror), remove the guard cell, then *Run all* on a T4 GPU (or CPU).
- 著者による学習コード未公開、ライセンスファイルなし。Drive を公開設定にすれば本ノートブックで検証可能です。